# 3D Kooshball GRE: simulation and self-supervised DeepADMM reconstruction

Pipeline: (1) build a 3D radial ("kooshball") spoiled GRE sequence with Pulseq,
(2) simulate the multi-coil signal on a BrainWeb phantom, (3) check the data with a
density-compensated adjoint NUFFT, (4) reconstruct with conjugate gradient (CG) and
with a self-supervised, learned ADMM (DeepADMM).

## Imports and device

All tensors are created on the GPU by default.

In [ ]:
%matplotlib widget

import itertools
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
import torch.nn as nn
import torchkbnufft as tkbn
from torch.utils.checkpoint import checkpoint
from tqdm import tqdm

from Modules import default_system, load_brain_phantom
from Modules.CoilSens import cyclic_head_coil
from Modules.fast_iso import isochromat_sim_fast
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Plan import KbNufft, KbNufftAdjoint
from Modules.Regularizer import Regularizer

torch.set_default_device("cuda")
device = torch.device("cuda")

## Sequence design

3D radial spoiled GRE with RF spoiling. Spoke directions follow a golden-angle
spiral on the sphere (polar angle from an equal-area spacing in $\cos\phi$);
`n_dummy` dummy shots along x (no ADC) bring the magnetisation to steady state.

In [ ]:
def main(
    test_report: bool = False,
    fov: float = 250e-3,
    n_x: int = 64,
    flip_angle_deg: float = 5,
    slice_thickness: float = 3e-3,
    echo_time: float = 3.98e-3,
    repetition_time: float = 100e-3,
    n_spokes: int = 8000,
    n_dummy: int = 20,
    rf_spoiling_increment_deg: float = 117,
    write_seq: bool = False,
) -> None:
    """Build, check and optionally write the 3D radial GRE sequence.

    Parameters
    ----------
    test_report : bool, optional
        Currently unused.
    fov : float, optional
        Field of view in m.
    n_x : int, optional
        Number of readout samples per spoke.
    flip_angle_deg : float, optional
        Excitation flip angle in degrees.
    slice_thickness : float, optional
        Excitation slab thickness in m.
    echo_time : float, optional
        Echo time in s.
    repetition_time : float, optional
        Repetition time in s.
    n_spokes : int, optional
        Number of acquired spokes.
    n_dummy : int, optional
        Number of dummy shots (no ADC) before the first acquired spoke.
    rf_spoiling_increment_deg : float, optional
        Quadratic RF-spoiling phase increment in degrees.
    write_seq : bool, optional
        Write the sequence to ``3D Kooshball GRE.seq``.

    Returns
    -------
    None
    """
    system = default_system()
    seq = pp.Sequence(system)
    delta_k = 1 / fov

    rf, grad_slice, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=flip_angle_deg * np.pi / 180,
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
    )

    grad_readout_base = pp.make_trapezoid(
        channel="x", flat_area=n_x * delta_k, flat_time=6.4e-3 / 5, system=system
    )
    adc = pp.make_adc(
        num_samples=n_x,
        duration=grad_readout_base.flat_time,
        delay=grad_readout_base.rise_time,
        system=system,
    )
    grad_prephase_base = pp.make_trapezoid(
        channel="x", area=-grad_readout_base.area / 2, duration=2e-3, system=system
    )
    grad_slice_rephase = pp.make_trapezoid(
        channel="z", area=-grad_slice.area / 2, duration=2e-3, system=system
    )
    grad_spoil_base = pp.make_trapezoid(
        channel="x", area=0.5 * n_x * delta_k, system=system
    )
    grad_slice_spoil = pp.make_trapezoid(
        channel="z", area=4 / slice_thickness, system=system
    )

    raster = seq.grad_raster_time
    delay_te = (
        np.ceil(
            (
                echo_time
                - pp.calc_duration(grad_prephase_base)
                - grad_slice.fall_time
                - grad_slice.flat_time / 2
                - pp.calc_duration(grad_readout_base) / 2
            )
            / raster
        )
        * raster
    )
    delay_tr = (
        np.ceil(
            (
                repetition_time
                - pp.calc_duration(grad_prephase_base)
                - pp.calc_duration(grad_slice)
                - pp.calc_duration(grad_readout_base)
                - delay_te
            )
            / raster
        )
        * raster
    )
    assert delay_tr > pp.calc_duration(grad_spoil_base, grad_slice_spoil)
    rf_phase_deg = 0
    rf_inc_deg = 0

    for i in range(n_spokes + n_dummy):
        rf.phase_offset = rf_phase_deg / 180 * np.pi
        adc.phase_offset = rf_phase_deg / 180 * np.pi

        rf_inc_deg = (rf_inc_deg + rf_spoiling_increment_deg) % 360
        rf_phase_deg = (rf_phase_deg + rf_inc_deg) % 360

        spoke_idx = i - n_dummy
        if spoke_idx >= 0:
            polar_angle = np.arccos(1 - 2 * (spoke_idx + 0.5) / n_spokes)
            azimuth_angle = np.pi * (1 + np.sqrt(5)) * spoke_idx

            dir_x = np.sin(polar_angle) * np.cos(azimuth_angle)
            dir_y = np.sin(polar_angle) * np.sin(azimuth_angle)
            dir_z = np.cos(polar_angle)
        else:
            dir_x, dir_y, dir_z = 1, 0, 0

        seq.add_block(rf, grad_slice)

        grad_prephase_x = pp.make_trapezoid(
            channel="x",
            area=grad_prephase_base.area * dir_x,
            duration=2e-3,
            system=system,
        )
        grad_prephase_y = pp.make_trapezoid(
            channel="y",
            area=grad_prephase_base.area * dir_y,
            duration=2e-3,
            system=system,
        )
        grad_prephase_z = pp.make_trapezoid(
            channel="z",
            area=grad_prephase_base.area * dir_z + grad_slice_rephase.area,
            duration=2e-3,
            system=system,
        )

        seq.add_block(grad_prephase_x, grad_prephase_y, grad_prephase_z)
        seq.add_block(pp.make_delay(delay_te))

        grad_readout_x = pp.make_trapezoid(
            channel="x",
            amplitude=grad_readout_base.amplitude * dir_x,
            flat_time=grad_readout_base.flat_time,
            rise_time=grad_readout_base.rise_time,
            fall_time=grad_readout_base.fall_time,
            system=system,
        )
        grad_readout_y = pp.make_trapezoid(
            channel="y",
            amplitude=grad_readout_base.amplitude * dir_y,
            flat_time=grad_readout_base.flat_time,
            rise_time=grad_readout_base.rise_time,
            fall_time=grad_readout_base.fall_time,
            system=system,
        )
        grad_readout_z = pp.make_trapezoid(
            channel="z",
            amplitude=grad_readout_base.amplitude * dir_z,
            flat_time=grad_readout_base.flat_time,
            rise_time=grad_readout_base.rise_time,
            fall_time=grad_readout_base.fall_time,
            system=system,
        )

        if spoke_idx >= 0:
            seq.add_block(grad_readout_x, grad_readout_y, grad_readout_z, adc)
        else:
            seq.add_block(grad_readout_x, grad_readout_y, grad_readout_z)

        grad_spoil_x = pp.make_trapezoid(
            channel="x", area=grad_spoil_base.area * dir_x, system=system
        )
        grad_spoil_y = pp.make_trapezoid(
            channel="y", area=grad_spoil_base.area * dir_y, system=system
        )
        grad_spoil_z = pp.make_trapezoid(
            channel="z",
            area=grad_spoil_base.area * dir_z + grad_slice_spoil.area,
            system=system,
        )

        seq.add_block(
            grad_spoil_x,
            grad_spoil_y,
            grad_spoil_z,
            pp.make_delay(delay_tr),
        )

    timing_ok, error_report = seq.check_timing()
    if timing_ok:
        print("Timing check passed successfully")
    else:
        print("Timing check failed! Error listing follows:")
        print(error_report)

    if write_seq:
        seq.set_definition("FOV", [fov, fov, slice_thickness])
        seq.set_definition("Name", "gre_rad")
        seq.write("3D Kooshball GRE.seq")

## Parameters

In [ ]:
fov = 250e-3
n_x = 128  # phantom + recon matrix (n_x^3); samples per spoke so kmax matches the grid
n_spokes = 5700  # ~ (pi/2) * n_x^2 spokes for full sampling
n_channels = 8
spin_count = 64  # isochromats per voxel

## Build the sequence

In [ ]:
main(fov=fov, n_x=n_x, n_spokes=n_spokes, write_seq=True)
mr_seq = mr0.Sequence.import_file("3D Kooshball GRE.seq")

## Phantom and coil sensitivities

BrainWeb phantom resampled to an $n_x^3$ grid, with cyclic head-coil sensitivities
scaled to the physical FOV.

In [ ]:
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n_x, n_x, n_x),
    device=device,
    permute_order=(2, 1, 0),
)

scale = tuple(float(s) / fov for s in phantom.size)  # (1, 1, 1) for a FOV cube
coil_sim = cyclic_head_coil((n_channels, n_x, n_x, n_x), fov_scaling=scale).to(
    torch.complex64
)
phantom.coil_sens = coil_sim.to(device)
voxel_data = phantom.build()

## Signal simulation

Ground-truth isochromat simulation of the sequence; the signal has shape
`(n_channels, n_adc_samples)`.

In [ ]:
torch.manual_seed(0)
signal = isochromat_sim_fast(
    mr_seq,
    voxel_data.cuda(),
    spin_count=spin_count,
    spin_dist="rand",
    print_progress=True,
)
signal = signal.permute((1, 0)).contiguous().to(torch.complex64)

## Helper functions for the adjoint-NUFFT check

The ADC k-space is converted to rad/pixel, reconstructed per channel with a
density-compensated adjoint NUFFT, and the simulated coil maps are matched to the
orientation of the resulting images (best of the 48 axis permutations/flips).

In [ ]:
def kspace_to_rad(mr_seq: mr0.Sequence, n_x: int, fov: float) -> torch.Tensor:
    """Convert the ADC k-space (1/m) to rad/pixel of an ``n_x**3`` grid.

    Parameters
    ----------
    mr_seq : mr0.Sequence
        Sequence providing the ADC k-space.
    n_x : int
        Matrix size of the reconstruction grid.
    fov : float
        Field of view in m.

    Returns
    -------
    torch.Tensor
        k-space in rad/pixel, shape ``(3, n_adc_samples)``.
    """
    k = torch.as_tensor(mr_seq.get_kspace()).float()
    k = k.reshape(-1, k.shape[-1])[:, :3].T.contiguous()
    return k * (2 * np.pi * fov / n_x)


def nufft_recon(signal: torch.Tensor, k_rad: torch.Tensor, n_x: int) -> torch.Tensor:
    """Reconstruct each channel with a density-compensated adjoint NUFFT.

    Samples beyond the kmax of the grid are dropped (not clamped).

    Parameters
    ----------
    signal : torch.Tensor
        Multi-coil k-space data, shape ``(n_channels, n_samples)``.
    k_rad : torch.Tensor
        k-space trajectory in rad/pixel, shape ``(3, n_samples)``.
    n_x : int
        Matrix size of the reconstruction grid.

    Returns
    -------
    torch.Tensor
        Complex images of shape ``(n_channels, n_x, n_x, n_x)``.
    """
    k_rad = k_rad.to(signal.device)
    inside_kmax = k_rad.norm(dim=0) <= np.pi
    k_kept, signal_kept = k_rad[:, inside_kmax], signal[:, inside_kmax]
    # |k|^2 density compensation for radial sampling.
    density_comp = k_kept.norm(dim=0).clamp(min=np.pi / n_x) ** 2
    adjoint_nufft = tkbn.KbNufftAdjoint(
        im_size=(n_x, n_x, n_x), grid_size=(2 * n_x,) * 3
    ).to(signal.device)
    weighted = signal_kept * density_comp
    batch = weighted[None]
    return adjoint_nufft(batch, k_kept)[0]


def relative_profile(volume: np.ndarray) -> np.ndarray:
    """Normalise channel magnitudes by their root-sum-of-squares.

    Parameters
    ----------
    volume : np.ndarray
        Complex multi-coil volume, shape ``(n_channels, n_x, n_x, n_x)``.

    Returns
    -------
    np.ndarray
        Relative channel magnitudes of the same shape.
    """
    magnitude = np.abs(volume)
    rss = np.sqrt((magnitude**2).sum(0, keepdims=True))
    return magnitude / (rss + 1e-12)


def permute_and_flip(
    volume: np.ndarray, perm: Tuple[int, ...], flips: Tuple[bool, ...]
) -> np.ndarray:
    """Permute and flip the three spatial axes of a multi-coil volume.

    Parameters
    ----------
    volume : np.ndarray
        Volume of shape ``(n_channels, n_x, n_x, n_x)``.
    perm : Tuple[int, ...]
        Permutation of the spatial axes ``(0, 1, 2)``.
    flips : Tuple[bool, ...]
        Whether to flip each spatial axis after permuting.

    Returns
    -------
    np.ndarray
        Permuted and flipped volume.
    """
    permuted = volume.transpose(0, *[p + 1 for p in perm])
    flip_axes = tuple(axis + 1 for axis, flip in enumerate(flips) if flip)
    return np.flip(permuted, flip_axes) if flip_axes else permuted


def match_coil_orientation(
    coil: torch.Tensor, img: torch.Tensor, mask_frac: float = 0.3
) -> Tuple[np.ndarray, Tuple[int, ...], Tuple[bool, ...], float]:
    """Find the axis permutation/flip of the coil maps that matches the images.

    Compares ``|coil_c| / rss(coil)`` with ``|img_c| / rss(img)`` inside the head
    for all 48 axis permutations and flips of ``coil``.

    Parameters
    ----------
    coil : torch.Tensor
        Coil sensitivities, shape ``(n_channels, n_x, n_x, n_x)``.
    img : torch.Tensor
        Raw NUFFT images, shape ``(n_channels, n_x, n_x, n_x)``.
    mask_frac : float, optional
        Fraction of the maximum root-sum-of-squares image defining the head mask.

    Returns
    -------
    coil_matched : np.ndarray
        Coil maps in the orientation of the images.
    perm : Tuple[int, ...]
        Best axis permutation.
    flips : Tuple[bool, ...]
        Best axis flips.
    score : float
        Correlation of the best match (close to 1 for a good match).
    """
    coil, img = np.asarray(coil), np.asarray(img)
    rss = np.sqrt((np.abs(img) ** 2).sum(0))
    mask = rss > mask_frac * rss.max()
    ref = relative_profile(img)[:, mask].ravel()
    ref = ref - ref.mean()

    best_score, best_perm, best_flips = -2.0, None, None
    for perm in itertools.permutations((0, 1, 2)):
        for flips in itertools.product([False, True], repeat=3):
            candidate = permute_and_flip(coil, perm, flips)
            profile = relative_profile(candidate)[:, mask].ravel()
            profile = profile - profile.mean()
            norm_product = np.linalg.norm(ref) * np.linalg.norm(profile) + 1e-12
            score = float(ref @ profile / norm_product)
            if score > best_score:
                best_score, best_perm, best_flips = score, perm, flips

    coil_matched = np.ascontiguousarray(permute_and_flip(coil, best_perm, best_flips))
    return coil_matched, best_perm, best_flips, best_score

## Adjoint-NUFFT check

Compare the root-sum-of-squares image, the sensitivity-weighted coil combination
and a single channel at the centre slice.

In [ ]:
k_rad = kspace_to_rad(mr_seq, n_x, fov)
assert k_rad.shape[1] == signal.shape[1], (k_rad.shape, signal.shape)  # ADC order

img = nufft_recon(signal, k_rad, n_x)  # (n_channels, n_x, n_x, n_x), raw orientation

coil_np, perm, flips, score = match_coil_orientation(
    coil_sim.cpu().numpy(), img.cpu().numpy()
)
print(f"coil orientation: perm={perm} flips={flips} score={score:.3f}")  # want ~1
coil_recon = torch.from_numpy(coil_np).to(img.device)

sens_power = coil_recon.abs().square().sum(0)
combined = (coil_recon.conj() * img).sum(0) / sens_power.clamp(
    min=1e-2 * sens_power.max()
)
rss = img.abs().square().sum(0).sqrt()

mid = n_x // 2


def show(ax: plt.Axes, vol: torch.Tensor, title: str) -> None:
    """Show the centre z-slice of a complex volume (magnitude, rotated by 90 degrees).

    Parameters
    ----------
    ax : plt.Axes
        Axes to draw on.
    vol : torch.Tensor
        Complex volume of shape ``(n_x, n_x, n_x)``.
    title : str
        Axes title.

    Returns
    -------
    None
    """
    ax.imshow(
        torch.rot90(vol.abs(), k=1, dims=(0, 1))[:, :, mid].cpu().numpy(), cmap="gray"
    )
    ax.set_title(title)
    ax.axis("off")


fig, ax = plt.subplots(1, 3, figsize=(12, 4))
show(ax[0], rss, "RSS (no coil maps)")
show(ax[1], combined, "sens-weighted combination")
show(ax[2], img[0], "channel 0 alone")
plt.show()

## k-space trajectory for the iterative reconstruction

Read the ADC trajectory from the sequence file and normalise it to
$[-0.5, 0.5]$. The simulated phantom data and trajectory are also saved to disk
for later reconstruction experiments.

In [ ]:
seq = pp.Sequence()
seq.read("3D Kooshball GRE.seq")
k_data = signal.clone().contiguous()
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:3, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

torch.save(
    {"data": voxel_data.cpu(), "k": k_traj_adc.cpu(), "coil": coil_sim.cpu()},
    "recon_inputs.pt",
)

## Linear operators

Complex tensors are handled as flattened real vectors (`view_as_real`), so the
operators act on real vectors of twice the length. `ChannelOperator` maps an image
to multi-coil images, `NonuniformFourierTransformOperator` maps multi-coil images
to non-Cartesian k-space samples, and `IdentityOperator` is used by ADMM.

In [ ]:
class ChannelOperator(LinearOperator):
    """Coil sensitivity operator: image to multi-coil images.

    Parameters
    ----------
    coil_sensitivities : torch.Tensor
        Complex coil sensitivities of shape ``(n_channels, *data_shape)``.
    data_shape : Tuple[int, ...]
        Spatial shape of the image.
    device : Optional[torch.device], optional
        Device attribute of the operator.
    """

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        n_channels = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (n_channels, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the operator: image to multi-coil images.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil images.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint: multi-coil images to coil-combined image.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil images.

        Returns
        -------
        torch.Tensor
            Real-valued flattened coil-combined image.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class NonuniformFourierTransformOperator(LinearOperator):
    """Non-Cartesian Fourier operator: multi-coil 3D images to k-space samples.

    Parameters
    ----------
    k : torch.Tensor
        k-space trajectory in cycles/voxel, shape ``(3, n_samples)``. The sign is
        flipped internally to match the NUFFT convention.
    input_shape : Tuple[int, ...]
        Shape ``(n_channels, n_x, n_y, n_z)`` of the multi-coil images.
    device : Optional[torch.device], optional
        Device on which the NUFFT objects are placed.
    """

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        _, n_samples = k.shape
        n_channels, n_x, n_y, n_z = input_shape
        self.n_modes = (n_x, n_y, n_z)
        self.k = (-k).unsqueeze(-1)  # (3, n_samples, 1)
        self.forward_shape = (n_channels, n_x, n_y, n_z, 1)
        self.adjoint_shape = (n_channels, n_samples)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)
        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the forward NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened multi-coil images.

        Returns
        -------
        torch.Tensor
            Real-valued flattened k-space samples.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the adjoint NUFFT.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened k-space samples.

        Returns
        -------
        torch.Tensor
            Real-valued flattened multi-coil images.
        """
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class IdentityOperator(LinearOperator):
    """Identity operator on real-valued views of complex images.

    Parameters
    ----------
    data_shape : Tuple[int, ...]
        Shape of the (complex) image.
    device : Optional[torch.device], optional
        Device attribute of the operator.
    """

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ) -> None:
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        """Return ``x`` unchanged.

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image.

        Returns
        -------
        torch.Tensor
            The input.
        """
        return x

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        """Return ``x`` unchanged (the operator is self-adjoint).

        Parameters
        ----------
        x : torch.Tensor
            Real-valued flattened image.

        Returns
        -------
        torch.Tensor
            The input.
        """
        return x

## DeepADMM

Solves $\min_x \tfrac{1}{2}\|Ax - d\|_2^2 + R(x)$ by ADMM, where the $x$-update is
a few CG iterations on $(Q + \rho I)\,x = b + \rho\,(z - u)$ with
$Q = A^H A$ and $b = A^H d$, and the $z$-update applies the learned regularizer.
$\rho$ is trainable. During training each ADMM iteration is gradient-checkpointed
to save memory.

In [ ]:
class DeepADMM(nn.Module):
    """ADMM solver with a learned 3D regularizer and a trainable penalty weight.

    Parameters
    ----------
    shape : Tuple[int, ...]
        Shape of the image the regularizer and identity operator act on.
    rho_init : float, optional
        Initial value of the ADMM penalty parameter ``rho``.
    maxiter_admm : int, optional
        Number of ADMM iterations.
    maxiter_cg : int, optional
        Number of CG iterations per ADMM ``x``-update.
    kernel_size : Tuple[int, ...], optional
        Convolution kernel size of the regularizer.
    features : int, optional
        Number of feature channels of the regularizer.
    num_of_resblocks : int, optional
        Number of residual blocks of the regularizer.
    scale_factor : int, optional
        Scale factor of the regularizer.
    checkpoint_iterations : bool, optional
        Use gradient checkpointing for each ADMM iteration while training.
    """

    def __init__(
        self,
        shape: Tuple[int, ...],
        rho_init: float = 0.1,
        maxiter_admm: int = 10,
        maxiter_cg: int = 5,
        kernel_size: Tuple[int, ...] = (3, 3, 3),
        features: int = 16,
        num_of_resblocks: int = 4,
        scale_factor: int = 1,
        checkpoint_iterations: bool = True,
    ) -> None:
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(
            shape,
            kernel_size=kernel_size,
            features=features,
            num_of_resblocks=num_of_resblocks,
            scale_factor=scale_factor,
        )
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg
        self.checkpoint_iterations = checkpoint_iterations

    def _step(
        self,
        x: torch.Tensor,
        z: torch.Tensor,
        u: torch.Tensor,
        Q: LinearOperator,
        b: torch.Tensor,
        disable_progressbar: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Perform one ADMM iteration.

        Parameters
        ----------
        x : torch.Tensor
            Current image estimate (real-valued view).
        z : torch.Tensor
            Current auxiliary variable.
        u : torch.Tensor
            Current scaled dual variable.
        Q : LinearOperator
            Normal operator ``A^H A``.
        b : torch.Tensor
            Right-hand side ``A^H d``.
        disable_progressbar : bool, optional
            Disable the CG progress bar.

        Returns
        -------
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor]
            Updated ``x``, ``z`` and ``u``.
        """
        x = conjugate_gradient(
            Q + self.rho * self.I,
            b=b + self.rho * (z - u),
            x=x,
            maxiter=self.maxiter_cg,
            disable_progressbar=disable_progressbar,
        )
        z = self.regularizer(x + u)
        u = u + x - z
        return x, z, u

    def forward(
        self,
        Q: LinearOperator,
        b: torch.Tensor,
        x: torch.Tensor,
        disable_progressbar: bool = True,
    ) -> torch.Tensor:
        """Run the unrolled ADMM iterations.

        Parameters
        ----------
        Q : LinearOperator
            Normal operator ``A^H A``.
        b : torch.Tensor
            Right-hand side ``A^H d`` (real-valued view of the complex image).
        x : torch.Tensor
            Initial image estimate (real-valued view).
        disable_progressbar : bool, optional
            Disable the CG progress bar.

        Returns
        -------
        torch.Tensor
            Reconstructed image (real-valued view).
        """
        z = torch.zeros(self.I.shape[0], dtype=x.dtype)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)
        use_checkpoint = (
            self.checkpoint_iterations and self.training and torch.is_grad_enabled()
        )
        for _ in range(self.maxiter_admm):
            if use_checkpoint:
                x, z, u = checkpoint(
                    self._step, x, z, u, Q, b, disable_progressbar, use_reentrant=False
                )
            else:
                x, z, u = self._step(x, z, u, Q, b, disable_progressbar)
        return x

## CG reconstruction

Notation: $F$ is the NUFFT, $C$ the coil sensitivities, $A = F C$ the encoding
operator, $Q = A^H A$ and $b = A^H d$. We solve $Q x = b$ with 30 CG iterations
starting from zero.

In [ ]:
F = NonuniformFourierTransformOperator(k_traj_adc, (n_channels, n_x, n_x, n_x))
C = ChannelOperator(coil_sim, (n_x, n_x, n_x))
A = F @ C

In [ ]:
b = A.H @ k_data.view(torch.float32).ravel()
Q = A.H @ A
x_init = torch.zeros(n_x * n_x * n_x, dtype=torch.complex64).view(torch.float32)
x_recon = conjugate_gradient(Q, b, x_init, maxiter=30)
x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_x, n_x))

Helper to show the three orthogonal centre slices of a reconstructed volume.

In [ ]:
def plot_orthogonal_views(volume: torch.Tensor, title: str) -> None:
    """Plot the three orthogonal centre slices of a complex volume (magnitude).

    Parameters
    ----------
    volume : torch.Tensor
        Complex volume of shape ``(n_x, n_x, n_x)``.
    title : str
        Figure title.

    Returns
    -------
    None
    """
    vol = torch.abs(volume).cpu().numpy()
    mid = vol.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Slices through the center of the volume, rotated by 180 degrees.
    axes[0].imshow(np.rot90(vol[mid, :, :].T, 2), cmap="gray")
    axes[0].set_title("Plane 1 (X-slice)")

    axes[1].imshow(np.rot90(vol[:, mid, :].T, 2), cmap="gray")
    axes[1].set_title("Plane 2 (Y-slice)")

    axes[2].imshow(np.rot90(vol[:, :, mid].T, 2), cmap="gray")
    axes[2].set_title("Plane 3 (Z-slice)")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(title)
    plt.show()

In [ ]:
plot_orthogonal_views(x_recon_image, "CG Reconstruction")

## Self-supervised training

Each step randomly splits the k-space samples into two halves: the model
reconstructs from the source half, and the loss (relative L1) compares the
predicted k-space on the target half with the measured data.

In [ ]:
shape = (n_x, n_x, n_x, 1)

torch.manual_seed(0)
model = DeepADMM(
    shape,
    rho_init=0.1,
    maxiter_admm=4,
    maxiter_cg=25,
)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

n_samples = k_traj_adc.shape[1]
num_splits = 10

model.train()
losses = []

progress_bar = tqdm(range(num_splits), desc="Self-supervised splits")

for split in progress_bar:
    np.random.seed(split)

    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    index_src, index_tar = np.split(indices, 2)

    k_traj_src = k_traj_adc[:, index_src]
    k_data_src = k_data[:, index_src].view(torch.float32).ravel()

    k_traj_tar = k_traj_adc[:, index_tar]
    k_data_tar = k_data[:, index_tar].view(torch.float32).ravel()

    F_src = NonuniformFourierTransformOperator(k_traj_src, (n_channels, n_x, n_x, n_x))
    F_tar = NonuniformFourierTransformOperator(k_traj_tar, (n_channels, n_x, n_x, n_x))

    Q_src = C.H @ F_src.H @ F_src @ C
    b_src = C.H @ F_src.H @ k_data_src

    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)

    k_data_pred = F_tar @ C @ x_out

    target_norm = torch.linalg.norm(k_data_tar, ord=1)
    loss = torch.linalg.norm(k_data_pred - k_data_tar, ord=1) / target_norm

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

In [ ]:
plt.figure()
plt.plot(losses, marker="o")
plt.xlabel("Split")
plt.ylabel("Self-supervised loss")
plt.title("DeepADMM training loss over splits")
plt.show()

## Trained reconstruction

Reconstruct from the full k-space data with the trained model.

In [ ]:
model.eval()

with torch.no_grad():
    x_deep_admm_trained = model(Q, b, b)

x_deep_admm_trained_image = x_deep_admm_trained.view(torch.complex64).reshape(
    (n_x, n_x, n_x)
)

In [ ]:
plot_orthogonal_views(
    x_deep_admm_trained_image, "DeepADMM (learned, trained) - Orthogonal Views"
)